# Poster Figures Notebook
Generates all figures for the QA-for-Sudoku poster session.

**Professor feedback addressed:**
- Fig 1: Sudoku puzzle visualization (the actual puzzles)
- Fig 2: One-hot encoding matrix replacing pie charts
- Fig 3: SA success rate bar chart (clean version)
- Fig 4: QUBO matrix heatmap (clean version)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.gridspec as gridspec

# Global style — clean, poster-ready
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 150,
})
print('Setup complete')

---
## Figure 1 — Sudoku Puzzle Visualization
Shows the actual 4×4 and 9×9 puzzles used in experiments.

In [ ]:
# ── Puzzle data ────────────────────────────────────────────────────
# 4×4 puzzle (0 = blank)
puzzle_4x4 = np.array([
    [0, 2, 0, 0],
    [0, 0, 0, 3],
    [2, 0, 0, 0],
    [0, 0, 4, 0],
])

# 9×9 Easy (30 givens)
puzzle_9x9_easy = np.array([
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9],
])

# 9×9 Hard (17 givens — minimum clue puzzle)
puzzle_9x9_hard = np.array([
    [8, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 3, 6, 0, 0, 0, 0, 0],
    [0, 7, 0, 0, 9, 0, 2, 0, 0],
    [0, 5, 0, 0, 0, 7, 0, 0, 0],
    [0, 0, 0, 0, 4, 5, 7, 0, 0],
    [0, 0, 0, 1, 0, 0, 0, 3, 0],
    [0, 0, 1, 0, 0, 0, 0, 6, 8],
    [0, 0, 8, 5, 0, 0, 0, 1, 0],
    [0, 9, 0, 0, 0, 0, 4, 0, 0],
])


def draw_sudoku(ax, grid, title, box_size):
    """Draw a sudoku puzzle grid on the given axes."""
    N = grid.shape[0]
    given_color = '#1a1a2e'
    given_bg    = '#e8f4f8'
    empty_bg    = '#ffffff'

    ax.set_xlim(0, N)
    ax.set_ylim(0, N)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=8)

    # Cell backgrounds
    for r in range(N):
        for c in range(N):
            val = grid[r, c]
            bg = given_bg if val != 0 else empty_bg
            rect = plt.Rectangle((c, N - 1 - r), 1, 1,
                                  facecolor=bg, edgecolor='#aaaaaa', linewidth=0.5)
            ax.add_patch(rect)
            if val != 0:
                ax.text(c + 0.5, N - 1 - r + 0.5, str(val),
                        ha='center', va='center',
                        fontsize=14 if N == 4 else 10,
                        fontweight='bold', color=given_color)

    # Thin cell lines
    for i in range(N + 1):
        lw = 0.5
        ax.axhline(i, color='#aaaaaa', linewidth=lw)
        ax.axvline(i, color='#aaaaaa', linewidth=lw)

    # Thick box lines
    for i in range(0, N + 1, box_size):
        ax.axhline(i, color='#2c3e50', linewidth=2.5)
        ax.axvline(i, color='#2c3e50', linewidth=2.5)

    # Given count
    n_given = int(np.sum(grid != 0))
    ax.text(N / 2, -0.6, f'{n_given} givens  →  {N*N*N - n_given*(N-1) - n_given} free variables',
            ha='center', va='top', fontsize=9, color='#555555', style='italic')


fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Sudoku Test Puzzles Used in Experiments', fontsize=15, fontweight='bold', y=1.02)

draw_sudoku(axes[0], puzzle_4x4,      '4×4 Puzzle\n(Baseline — 64 vars)', box_size=2)
draw_sudoku(axes[1], puzzle_9x9_easy, '9×9 Easy (30 givens)\n459 free variables', box_size=3)
draw_sudoku(axes[2], puzzle_9x9_hard, '9×9 Hard (17 givens)\n576 free variables', box_size=3)

plt.tight_layout()
plt.savefig('fig1_sudoku_puzzles.png', bbox_inches='tight', dpi=200)
plt.show()
print('Saved fig1_sudoku_puzzles.png')

---
## Figure 2 — One-Hot Encoding Matrix (replaces pie charts)
Shows the QUBO variable structure as a binary matrix instead of a pie chart.
Left: full grid. Right: given cells fixed → columns eliminated.

In [ ]:
FREE_VAL  = 0
ELIM_VAL  = 1
FIXED_VAL = 2

CMAP_FULL = LinearSegmentedColormap.from_list(
    'ohc', ['#c9daf8', '#b7b7b7', '#1c4587'], N=3)

FREE_RGB = np.array([201, 218, 248]) / 255


def build_display_matrix(grid):
    N = grid.shape[0]
    display = np.full((N * N, N), FREE_VAL, dtype=float)
    for r in range(N):
        for c in range(N):
            v = grid[r, c]; ci = r * N + c
            if v:
                display[ci, :] = ELIM_VAL
                display[ci, v - 1] = FIXED_VAL
    return display, grid.flatten() == 0


def add_matrix_grid(ax, n_rows, n_cols, row_group):
    for c in range(n_cols + 1):
        ax.axvline(c - 0.5, color='white', linewidth=0.5)
    for r in range(n_rows + 1):
        ax.axhline(r - 0.5, color='white', linewidth=0.5)
    for k in range(0, n_rows + 1, row_group):
        ax.axhline(k - 0.5, color='white', linewidth=2.0)


def plot_full(ax, display, grid, label):
    N = grid.shape[0]
    ax.imshow(display, aspect='auto', cmap=CMAP_FULL,
              vmin=FREE_VAL, vmax=FIXED_VAL, interpolation='nearest')
    add_matrix_grid(ax, N*N, N, row_group=N)
    ax.set_title(f'{label}\n{N*N}×{N} = {N*N*N} variables',
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Digit', fontsize=9)
    ax.set_ylabel('Cell index', fontsize=9)
    ax.set_xticks(range(N))
    ax.set_xticklabels([str(d+1) for d in range(N)], fontsize=7)
    ax.tick_params(axis='y', labelsize=7)


def plot_reduced(ax, free_mask, grid):
    N = grid.shape[0]
    n_free  = int(np.sum(free_mask))
    n_given = int(np.sum(~free_mask))
    pct     = 100 * n_given * N / (N*N*N)

    img = np.tile(FREE_RGB, (n_free, N, 1))
    ax.imshow(img, aspect='auto', interpolation='nearest')
    add_matrix_grid(ax, n_free, N, row_group=N)

    ax.set_title(f'After reduction  (–{pct:.0f}%)\n'
                 f'{n_free}×{N} = {n_free*N} variables',
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Digit', fontsize=9)
    ax.set_ylabel('Free cell index', fontsize=9)
    ax.set_xticks(range(N))
    ax.set_xticklabels([str(d+1) for d in range(N)], fontsize=7)
    ax.tick_params(axis='y', labelsize=7)


# 4×4 first (cells big enough to see clearly), then 9×9 Easy (real scale, 37% reduction)
# 9×9 Hard dropped — only 21% reduction, least dramatic of the three
puzzles = [
    (puzzle_4x4,      '4×4  (4 givens)'),
    (puzzle_9x9_easy, '9×9 Easy  (30 givens)'),
]

fig = plt.figure(figsize=(12, 8))
fig.suptitle('One-Hot Encoding Over Grid-Based Representation\n'
             'Each cell × digit  =  1 binary variable  —  given cells eliminate entire rows',
             fontsize=13, fontweight='bold', y=1.01)

gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.6, wspace=0.35)

for row_idx, (grid, lbl) in enumerate(puzzles):
    N = grid.shape[0]
    display, free_mask = build_display_matrix(grid)

    ax_full = fig.add_subplot(gs[row_idx, 0])
    ax_red  = fig.add_subplot(gs[row_idx, 1])

    plot_full(ax_full, display, grid, lbl)
    plot_reduced(ax_red, free_mask, grid)

legend_elements = [
    mpatches.Patch(facecolor='#c9daf8', edgecolor='#aaa', label='Free variable (0 or 1)'),
    mpatches.Patch(facecolor='#b7b7b7', edgecolor='#aaa', label='Eliminated  (given — wrong digit)'),
    mpatches.Patch(facecolor='#1c4587', edgecolor='#aaa', label='Fixed = 1  (given — correct digit)'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=9,
           bbox_to_anchor=(0.5, -0.04), framealpha=0.9)

plt.savefig('fig2_onehot_matrix.png', bbox_inches='tight', dpi=200)
plt.show()
print('Saved fig2_onehot_matrix.png')

---
## Figure 3 — SA Success Rate Bar Chart (clean poster version)

In [ ]:
labels      = ['4×4\n(64 vars)', '9×9 Easy\n(459 vars)', '9×9 Medium\n(504 vars)', '9×9 Hard\n(576 vars)']
success_pct = [99, 6, 0, 0]
colors      = ['#2ecc71' if s > 50 else ('#f39c12' if s > 0 else '#e74c3c') for s in success_pct]

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(labels, success_pct, color=colors, edgecolor='white', linewidth=1.5,
              width=0.55, zorder=3)

# Value labels on bars
for bar, val in zip(bars, success_pct):
    if val > 0:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 1.5,
                f'{val}%',
                ha='center', va='bottom', fontsize=13, fontweight='bold',
                color='#2c3e50')
    else:
        ax.text(bar.get_x() + bar.get_width() / 2,
                2, '0%',
                ha='center', va='bottom', fontsize=13, fontweight='bold',
                color='#e74c3c')

ax.set_ylim(0, 115)
ax.set_ylabel('Success Rate (%)', fontsize=12)
ax.set_title('Simulated Annealing — Success Rate\n(100 independent samples per puzzle)',
             fontsize=13, fontweight='bold')
ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax.set_axisbelow(True)
ax.spines[['top', 'right']].set_visible(False)

# Annotation about variable density
ax.annotate('14× variable density increase\ncollapses success from 99% → 0%',
            xy=(1, 6), xytext=(2.5, 55),
            fontsize=9, color='#555555', style='italic',
            arrowprops=dict(arrowstyle='->', color='#888888', lw=1.2))

plt.tight_layout()
plt.savefig('fig3_sa_success_rate.png', bbox_inches='tight', dpi=200)
plt.show()
print('Saved fig3_sa_success_rate.png')

---
## Figure 4 — 4×4 QUBO Matrix (sparsity pattern + heatmap)

In [ ]:
def build_sudoku_qubo_4x4(puzzle):
    """
    Build 64×64 QUBO matrix for a 4×4 Sudoku.
    Variables: x[i,j,k] — index = (i*4 + j)*4 + k
    Penalties: E1 cell, E2 row, E3 col, E4 box.
    Lambda = 1 for all.
    """
    N = 4
    n_vars = N * N * N  # 64
    Q = np.zeros((n_vars, n_vars))

    def idx(i, j, k):  # cell (i,j), digit k (0-indexed)
        return (i * N + j) * N + k

    lam = 1.0

    # ── E1: one digit per cell ─────────────────────────────────────
    for i in range(N):
        for j in range(N):
            for k in range(N):
                Q[idx(i,j,k), idx(i,j,k)] += -lam
            for k1 in range(N):
                for k2 in range(k1+1, N):
                    Q[idx(i,j,k1), idx(i,j,k2)] += 2*lam

    # ── E2: one digit per row ──────────────────────────────────────
    for i in range(N):
        for k in range(N):
            for j1 in range(N):
                Q[idx(i,j1,k), idx(i,j1,k)] += -lam
                for j2 in range(j1+1, N):
                    Q[idx(i,j1,k), idx(i,j2,k)] += 2*lam

    # ── E3: one digit per col ──────────────────────────────────────
    for j in range(N):
        for k in range(N):
            for i1 in range(N):
                Q[idx(i1,j,k), idx(i1,j,k)] += -lam
                for i2 in range(i1+1, N):
                    Q[idx(i1,j,k), idx(i2,j,k)] += 2*lam

    # ── E4: one digit per √N×√N box ───────────────────────────────
    box = 2
    for br in range(box):
        for bc in range(box):
            cells = [(br*box + r, bc*box + c) for r in range(box) for c in range(box)]
            for k in range(N):
                for ci, (i1, j1) in enumerate(cells):
                    Q[idx(i1,j1,k), idx(i1,j1,k)] += -lam
                    for (i2, j2) in cells[ci+1:]:
                        Q[idx(i1,j1,k), idx(i2,j2,k)] += 2*lam

    return Q


Q = build_sudoku_qubo_4x4(puzzle_4x4)

nonzero = np.sum(Q != 0)
total   = Q.shape[0] ** 2
sparsity = 100 * (1 - nonzero / total)
print(f'QUBO size: {Q.shape[0]}×{Q.shape[1]}')
print(f'Non-zero entries: {nonzero} / {total}  ({sparsity:.1f}% sparse)')

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle('4×4 Sudoku QUBO Matrix  (64×64)', fontsize=14, fontweight='bold')

# — Sparsity pattern
ax = axes[0]
ax.spy(Q != 0, markersize=1.8, color='#2c7bb6')
ax.set_title(f'Nonzero Pattern  (Sparsity: {sparsity:.1f}%)', fontsize=12)
ax.set_xlabel('Variable Index')
ax.set_ylabel('Variable Index')

# — Heatmap
ax = axes[1]
cmap = LinearSegmentedColormap.from_list('qubo', ['#1a3a6b', '#ffffff', '#8b1a1a'])
vmax = max(abs(Q.min()), abs(Q.max()))
im = ax.imshow(Q, cmap=cmap, vmin=-vmax, vmax=vmax, interpolation='nearest')
plt.colorbar(im, ax=ax, label='QUBO Coefficient', fraction=0.046, pad=0.04)
ax.set_title('Coefficient Values', fontsize=12)
ax.set_xlabel('Variable Index')
ax.set_ylabel('Variable Index')

# Box-block dividers (every 16 = one row/col group)
for a in [axes[0], axes[1]]:
    for k in range(0, 65, 16):
        a.axhline(k - 0.5, color='black', linewidth=0.8, alpha=0.4)
        a.axvline(k - 0.5, color='black', linewidth=0.8, alpha=0.4)

plt.tight_layout()
plt.savefig('fig4_qubo_matrix.png', bbox_inches='tight', dpi=200)
plt.show()
print('Saved fig4_qubo_matrix.png')

---
## Figure 5 — D-Wave Hybrid vs SA Comparison Table (optional formatted version)
The table data is already in the poster; this cell produces a clean matplotlib table figure.

In [ ]:
import matplotlib

table_data = [
    ['Easy (30 clues)',   '459', '6% ✓', '0 ✓', '0 ✓', '1.64s'],
    ['Medium (25 clues)', '504', '0% ✗', '4 ✗', '0 ✓', '1.80s'],
    ['Hard (17 clues)',   '576', '0% ✗', '4 ✗', '4 ✗', '1.72s'],
]
col_labels = ['Puzzle', 'Vars', 'SA', 'SA Energy', 'Hybrid Energy', 'Time']

fig, ax = plt.subplots(figsize=(10, 2.8))
ax.axis('off')
ax.set_title('D-Wave Hybrid Solver vs. Simulated Annealing',
             fontsize=13, fontweight='bold', pad=12)

table = ax.table(
    cellText=table_data,
    colLabels=col_labels,
    cellLoc='center',
    loc='center',
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.0)

# Style header
for j in range(len(col_labels)):
    table[(0, j)].set_facecolor('#2c3e50')
    table[(0, j)].set_text_props(color='white', fontweight='bold')

# Colour code SA vs Hybrid outcomes
cell_colors = [
    [None, None, '#d5f5e3', '#fadbd8', '#d5f5e3', None],  # Easy
    [None, None, '#fadbd8', '#fadbd8', '#d5f5e3', None],  # Medium
    [None, None, '#fadbd8', '#fadbd8', '#fadbd8', None],  # Hard
]
for i, row_colors in enumerate(cell_colors):
    for j, color in enumerate(row_colors):
        if color:
            table[(i+1, j)].set_facecolor(color)

# Alternating row backgrounds
for i in range(1, len(table_data)+1):
    for j in range(len(col_labels)):
        if cell_colors[i-1][j] is None:
            table[(i,j)].set_facecolor('#f8f9f9' if i % 2 == 0 else 'white')

plt.tight_layout()
plt.savefig('fig5_comparison_table.png', bbox_inches='tight', dpi=200)
plt.show()
print('Saved fig5_comparison_table.png')

---
## Notes for poster design chat

Generated files:
- `fig1_sudoku_puzzles.png` → addresses professor's *"include the actual puzzles"* note
- `fig2_onehot_matrix.png` → replaces pie charts; shows one-hot encoding over grid-based representation
- `fig3_sa_success_rate.png` → clean version of the existing bar chart
- `fig4_qubo_matrix.png` → clean version of the existing QUBO heatmap
- `fig5_comparison_table.png` → formatted D-Wave vs SA table

**Quantum hardware photo** — needs to be sourced manually (D-Wave Advantage QPU).  
Suggested source: D-Wave's press kit or Wikipedia commons image of the Advantage system.